In [1]:
import jieba
import os
import re
import numpy as np
import jieba.posseg as psg
import networkx as nx
import pandas as pd
import math
os.chdir("D:/pythonProject5/5_wordNet/C03_word_pmi")

In [2]:
def get_stop_dict(file):
    content = open(file,encoding="utf-8")
    word_list = []
    for c in content:
        c = re.sub('\n|\r','',c)
        word_list.append(c)
    return word_list

In [3]:
def get_data(path):
    t = open(path,encoding="utf-8")
    data = t.read()
    t.close()
    return data

In [4]:
def get_wordlist(text,maxn,synonym_words,stop_words):
    synonym_origin = list(synonym_words['origin'])
    synonym_new = list(synonym_words['new'])
    flag_list = ['n','nz','vn']#a,形容词，v,形容词
    counts={}
    
    
    text_seg = psg.cut(text)
    for word_flag in text_seg:
        #word = re.sub("[^\u4e00-\u9fa5]","",word_flag.word)
        word = word_flag.word
        if word_flag.flag in flag_list and len(word)>1 and word not in stop_words:
            if word in synonym_origin:
                index = synonym_origin.index(word)
                word = synonym_new[index]
            counts[word]=counts.get(word,0)+1
            
    
    words= sorted(counts.items(),key=lambda x:x[1],reverse=True)
    words= list(dict(words).keys())[0:maxn]
    
    return words

In [5]:
def get_t_seg(topwords, text, synonym_words, stop_words):
    """
    对文本进行分词和关键词提取。
    
    根据给定的停用词、同义词替换和词性标注，从文本中提取出topwords中的关键词。
    
    参数:
    topwords: 一个列表，包含需要提取的关键词。
    text: 待处理的文本，以字符串形式提供。
    synonym_words: 一个字典，包含同义词替换的映射关系。
    stop_words: 一个列表，包含需要过滤掉的停用词。
    
    返回:
    text_lines_seg: 一个列表，包含每行文本处理后的关键词列表。
    word_docs: 一个字典，记录每个关键词在所有文本中出现的次数。
    """
    # 初始化一个字典，用于存储每个词在文本中出现的次数
    word_docs = {}
    
    # 提取同义词字典中的原始词和新词列表
    synonym_origin = list(synonym_words['origin'])
    synonym_new = list(synonym_words['new'])
    
    # 定义需要考虑的词性标记列表
    flag_list = ['n', 'nz', 'vn']
    
    # 初始化一个列表，用于存储每行文本的关键词列表
    text_lines_seg = []
    
    # 将文本按行分割
    text_lines = text.split("\n")
    for line in text_lines:
        # 初始化当前行的关键词列表
        t_seg = []
        # 对当前行进行分词
        text_seg = psg.cut(line)
        for word_flag in text_seg:
            # 提取词和词性标记
            word = word_flag.word
            # 判断词是否满足条件（词性标记在列表中、词长度大于1、不是停用词）
            if word_flag.flag in flag_list and len(word) > 1 and word not in stop_words:
                # 如果词在同义词原始列表中，将其替换为同义词新列表中的词
                if word in synonym_origin:
                    word = synonym_new[synonym_origin.index(word)]
                # 如果词是topwords中的词，添加到当前行的关键词列表中
                if word in topwords:
                    t_seg.append(word)
        # 去除关键词列表中的重复项
        t_seg = list(set(t_seg))
        # 更新每个关键词在所有文本中出现的次数
        for word in t_seg:
            word_docs[word] = word_docs.get(word, 0) + 1
        # 将当前行的关键词列表添加到结果列表中
        text_lines_seg.append(t_seg)
    return text_lines_seg, word_docs

In [6]:
def get_comatrix(text_lines_seg):
    comatrix = pd.DataFrame(np.zeros([len(topwords),len(topwords)]),columns=topwords,index=topwords)
    for t_seg in text_lines_seg:
        for i in range(len(t_seg)-1):
                for j in range(i+1,len(t_seg)):
                    comatrix.loc[t_seg[i],t_seg[j]]+=1
    for k in range(len(comatrix)):
        comatrix.iloc[k,k]=0
    return comatrix

In [7]:
def get_pmi(word1,word2,word_docs,co_matrix,n):
    pw1 = word_docs[word1]/n
    pw2 = word_docs[word2]/n
    pw1w2 = (co_matrix.loc[word1][word2]+co_matrix.loc[word2][word1])/n
    if pw1w2/(pw1*pw2)<=1.5: # 这里原来是0，由于节点太过复杂，故调整为此数值
        return 0
    else:
        pmi = math.log2(pw1w2/(pw1*pw2))
        return pmi

In [8]:
def get_net(copmi,topwords):
    g = nx.Graph()
    for i in range(len(topwords)-1):
        word = topwords[i]
        for j in range(i+1,len(topwords)):
            w=0
            word2 = topwords[j]
            w = copmi.loc[word][word2]+copmi.loc[word2][word]
            if w>0:
                g.add_edge(word,word2,weight=w)
    return g

In [9]:
#文件路径
dic_file = "./stop_dic/dict.txt"
stop_file = "./stop_dic/stopwords.txt"
data_path = "./data/data.txt"
synonym_file = "./stop_dic/synonym_list.xlsx"

In [10]:
#读取文件
data = get_data(data_path)
# stop_words = get_stop_dict(stop_file) # 旧版本
stop_words = get_stop_dict('D:/pythonProject5/stopwords/baidu_stopwords.txt')+get_stop_dict('D:/pythonProject5/stopwords/cn_stopwords.txt')+get_stop_dict('D:/pythonProject5/stopwords/hit_stopwords.txt')+get_stop_dict('D:/pythonProject5/stopwords/scu_stopwords.txt')+get_stop_dict('D:/pythonProject5/stopwords/own_stopwords.txt')
jieba.load_userdict(dic_file)
synonym_words = pd.read_excel(synonym_file)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\summer\AppData\Local\Temp\jieba.cache
Loading model cost 0.386 seconds.
Prefix dict has been built successfully.


In [11]:
#数据处理
n_topwords=200
topwords = get_wordlist(data,n_topwords,synonym_words,stop_words)

In [12]:
t_segs,word_docs = get_t_seg(topwords,data,synonym_words,stop_words)
n = len(t_segs)
co_matrix = get_comatrix(t_segs)

In [13]:
copmi = pd.DataFrame(np.zeros([len(topwords),len(topwords)]),columns=topwords,index=topwords)
for i in range(len(topwords)-1):
    word1 = topwords[i]
    for j in range(i+1,len(topwords)):
        word2 = topwords[j]
        copmi[word1][word2] = get_pmi(word1,word2,word_docs,co_matrix,n)

C:\Users\summer\AppData\Local\Temp\ipykernel_6632\998957605.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  copmi[word1][word2] = get_pmi(word1,word2,word_docs,co_matrix,n)


In [14]:
co_net =get_net(copmi,topwords)

In [15]:
nx.write_gexf(co_net,"./result/word_pmi.gexf")